In [1]:
import os
from pathlib import Path
from pypdf import PdfReader

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
DATA_PATH = Path("../data/documents")

documents = []

for pdf_file in DATA_PATH.glob("*.pdf"):
    reader = PdfReader(pdf_file)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            documents.append({
                "text": text,
                "source": pdf_file.name,
                "page": page_number
            })

print("Number of pages loaded:", len(documents))

Number of pages loaded: 340


In [3]:
print("Source:", documents[0]["source"])
print("Page:", documents[0]["page"])
print("\nText preview:")
print(documents[0]["text"][:500])

Source: 1404.7828v4.pdf
Page: 1

Text preview:
Deep Learning in Neural Networks: An Overview
Technical Report IDSIA-03-14 / arXiv:1404.7828 v4 [cs.NE] (88 pages, 888 references)
J¨urgen Schmidhuber
The Swiss AI Lab IDSIA
Istituto Dalle Molle di Studi sull’Intelligenza Artiﬁciale
University of Lugano & SUPSI
Galleria 2, 6928 Manno-Lugano
Switzerland
8 October 2014
Abstract
In recent years, deep artiﬁcial neural networks (including recurrent ones) have won numerous
contests in pattern recognition and machine learning. This historical survey co


In [4]:
chunks = []

chunk_size = 1000
overlap = 200

for doc in documents:
    text = doc["text"]

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]

        if chunk_text.strip():
            chunks.append({
                "text": chunk_text,
                "source": doc["source"],
                "page": doc["page"]
            })

        start += chunk_size - overlap

print("Number of chunks:", len(chunks))

Number of chunks: 1079


In [5]:
print(chunks[0]["text"])
print("\nSource:", chunks[0]["source"])
print("Page:", chunks[0]["page"])

Deep Learning in Neural Networks: An Overview
Technical Report IDSIA-03-14 / arXiv:1404.7828 v4 [cs.NE] (88 pages, 888 references)
J¨urgen Schmidhuber
The Swiss AI Lab IDSIA
Istituto Dalle Molle di Studi sull’Intelligenza Artiﬁciale
University of Lugano & SUPSI
Galleria 2, 6928 Manno-Lugano
Switzerland
8 October 2014
Abstract
In recent years, deep artiﬁcial neural networks (including recurrent ones) have won numerous
contests in pattern recognition and machine learning. This historical survey compactly summarises
relevant work, much of it from the previous millennium. Shallow and deep learners are distin-
guished by the depth of theircredit assignment paths, which are chains of possibly learnable, causal
links between actions and effects. I review deep supervised learning (also recapitulating the history
of backpropagation), unsupervised learning, reinforcement learning & evolutionary computation,
and indirect search for short programs encoding deep and large networks.
LATEX source: ht

## 2.2 Chunking Strategy

The documents were split into fixed-size chunks of 1000 characters with an overlap of 200 characters.

This overlap helps preserve context between consecutive chunks and reduces the chance of losing information when a topic continues across two chunks.

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\rag-assistant-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Arwa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Embeddings shape: (1079, 384)


In [7]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="../backend/data/vector_store"
)

collection = chroma_client.get_or_create_collection(
    name="documents"
)

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "source": chunk["source"],
            "page": chunk["page"]
        }
        for chunk in chunks
    ]
)

print("Vector store created successfully!")
print("Number of documents:", collection.count())

Vector store created successfully!
Number of documents: 1079


In [8]:
question = "What is deep learning?"

question_embedding = embedding_model.encode([question])[0]

results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

for i in range(3):
    print("----- Result", i + 1, "-----")
    print("Source:", results["metadatas"][0][i]["source"])
    print("Page:", results["metadatas"][0][i]["page"])
    print(results["documents"][0][i][:500])
    print()

----- Result 1 -----
Source: 1404.7828v4.pdf
Page: 1
Deep Learning in Neural Networks: An Overview
Technical Report IDSIA-03-14 / arXiv:1404.7828 v4 [cs.NE] (88 pages, 888 references)
J¨urgen Schmidhuber
The Swiss AI Lab IDSIA
Istituto Dalle Molle di Studi sull’Intelligenza Artiﬁciale
University of Lugano & SUPSI
Galleria 2, 6928 Manno-Lugano
Switzerland
8 October 2014
Abstract
In recent years, deep artiﬁcial neural networks (including recurrent ones) have won numerous
contests in pattern recognition and machine learning. This historical survey co

----- Result 2 -----
Source: 1404.7828v4.pdf
Page: 4
1 Introduction to Deep Learning (DL) in Neural Networks (NNs)
Which modiﬁable components of a learning system are responsible for its success or failure? What
changes to them improve performance? This has been called thefundamental credit assignment prob-
lem (Minsky, 1963). There are general credit assignment methods for universal problem solvers that
are time-optimal in various theoretic

In [9]:
def retrieve_documents(question, k=3):
    question_embedding = embedding_model.encode([question])[0]

    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=k
    )

    retrieved = []

    for i in range(k):
        retrieved.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "page": results["metadatas"][0][i]["page"]
        })

    return retrieved

In [10]:
question = "What is deep learning?"

retrieved = retrieve_documents(question)

for item in retrieved:
    print("Source:", item["source"])
    print("Page:", item["page"])
    print(item["text"][:300])
    print("-" * 50)

Source: 1404.7828v4.pdf
Page: 1
Deep Learning in Neural Networks: An Overview
Technical Report IDSIA-03-14 / arXiv:1404.7828 v4 [cs.NE] (88 pages, 888 references)
J¨urgen Schmidhuber
The Swiss AI Lab IDSIA
Istituto Dalle Molle di Studi sull’Intelligenza Artiﬁciale
University of Lugano & SUPSI
Galleria 2, 6928 Manno-Lugano
Switzerl
--------------------------------------------------
Source: 1404.7828v4.pdf
Page: 4
1 Introduction to Deep Learning (DL) in Neural Networks (NNs)
Which modiﬁable components of a learning system are responsible for its success or failure? What
changes to them improve performance? This has been called thefundamental credit assignment prob-
lem (Minsky, 1963). There are general credit
--------------------------------------------------
Source: 1404.7828v4.pdf
Page: 16
989, 1990a, 1998) to Neocognitron-
like, weight-sharing, convolutional neural layers (Sec. 5.4) with adaptive connections. This combi-
nation, augmented by Max-Pooling (MP, Sec. 5.11, 5.16), and sped

In [11]:
import ollama

response = ollama.chat(
    model="smollm2:135m",
    messages=[
        {
            "role": "user",
            "content": "What is deep learning?"
        }
    ]
)

print(response["message"]["content"])

Deep learning is an interdisciplinary field that combines artificial intelligence with machine learning. It's a subset of computer vision and natural language processing (NLP) techniques. Inspired by the structure of the human brain and the way it processes information, deep learning algorithms learn to recognize patterns within images or text data using complex neural networks.

In essence, deep learning achieves this by recursively applying layers in a network architecture that mimics the structure of the human brain's processing units. These layers are called "units," which are essentially thin sets of neurons with additional processing capabilities, such as convolution and pooling operations. By analyzing the input data and identifying patterns, the network can learn to recognize objects or images from their surroundings.

Deep learning has become a crucial component in various fields like computer vision, image recognition, machine learning, natural language processing (NLP), and 

In [16]:
def ask_rag(question, k=3):
    retrieved = retrieve_documents(question, k)

    context = ""

    for item in retrieved:
        context += (
            f"Source: {item['source']}, Page: {item['page']}\n"
            f"{item['text']}\n\n"
        )

    prompt = f"""
You are a document question-answering assistant.

Answer the question using ONLY the context provided below.

Do not use outside knowledge.
Do not invent information.
Give a short and direct answer.
If the context does not contain the answer, say:
"I could not find the answer in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    response = ollama.chat(
        model="smollm2:135m",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"].strip()

    sources = [
        {
            "source": item["source"],
            "page": item["page"]
        }
        for item in retrieved
    ]

    return answer, sources
    return answer, sources

In [13]:
question = "What is deep learning?"

answer, sources = ask_rag(question)

print("ANSWER:")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(f"- {source['source']} - Page {source['page']}")
    

ANSWER:
Deep Learning in Neural Networks (NNs) has a wide range of applications across various domains such as computer vision, natural language processing, image recognition, and speech recognition. It can be applied to tasks such as

1. Deep learning for image and audio classification, with the ability to predict what should be shown or not show based on context information
2. Machine translation between languages by using neural networks to translate text into a single coherent human-like speech
3. Image recognition of handwritten digits (i.e., digitally encoded pixels) through convolutional neural layers with deep learning architectures like MNIST
4. Speech analysis and recognition in natural language processing (NLP, e.g., Natural Language Toolkit, NLTK), for which CNNs have been highly successful using
5. Machine learning algorithms to classify handwritten digits into 32 categories
6. Supervised learning over large datasets of text from news articles with complex semantic meaning

## 2.4 Retrieval & Prompting

For each user question, the system retrieves the top-k relevant chunks from ChromaDB.

The retrieved chunks are then provided to the local Ollama language model as context. The model is instructed to answer only from the provided context and to state when the answer cannot be found in the documents.

In [17]:
questions = [
    "What is deep learning?",
    "What are neural networks?",
    "What are the main applications of deep learning?",
    "What is the fundamental credit assignment problem?",
    "How are convolutional neural networks used in deep learning?",
    "What is max-pooling?",
    "How is deep learning related to machine learning?",
    "What are recurrent neural networks?",
    "How is deep learning used in speech recognition?",
    "How is deep learning used in image recognition?"
]

print("Number of questions:", len(questions))

Number of questions: 10


In [18]:
for i, question in enumerate(questions, start=1):
    results = retrieve_documents(question, k=3)

    print(f"\n{'=' * 60}")
    print(f"Question {i}: {question}")

    for j, item in enumerate(results, start=1):
        print(f"\nResult {j}:")
        print(f"Source: {item['source']}")
        print(f"Page: {item['page']}")
        print(item["text"][:200].replace("\n", " "))


Question 1: What is deep learning?

Result 1:
Source: 1404.7828v4.pdf
Page: 1
Deep Learning in Neural Networks: An Overview Technical Report IDSIA-03-14 / arXiv:1404.7828 v4 [cs.NE] (88 pages, 888 references) J¨urgen Schmidhuber The Swiss AI Lab IDSIA Istituto Dalle Molle di St

Result 2:
Source: 1404.7828v4.pdf
Page: 4
1 Introduction to Deep Learning (DL) in Neural Networks (NNs) Which modiﬁable components of a learning system are responsible for its success or failure? What changes to them improve performance? This

Result 3:
Source: 1404.7828v4.pdf
Page: 16
989, 1990a, 1998) to Neocognitron- like, weight-sharing, convolutional neural layers (Sec. 5.4) with adaptive connections. This combi- nation, augmented by Max-Pooling (MP, Sec. 5.11, 5.16), and sped 

Question 2: What are neural networks?

Result 1:
Source: 1404.7828v4.pdf
Page: 55
Honavar, V . and Uhr, L. M. (1988). A network of neuron-like units that learns to perceive by gen- eration as well as reweighting of its links. In T

In [ ]:
answer, sources = ask_rag(
    "What is deep learning?"
)

print("ANSWER:")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(f"- {source['source']} - Page {source['page']}")

ANSWER:
The question asks about the basics of deep learning and its significance in AI research, specifically in Deep Learning (DL)
in Neural Networks (NNs). The answer provides a brief overview of what makes deep learning different from simple neural networks. It also mentions that CNNs are the most famous benchmark for
Deep Learning.

SOURCES:
- 1404.7828v4.pdf - Page 1
- 1404.7828v4.pdf - Page 4
- 1404.7828v4.pdf - Page 16


: 

## 2.5 CLI Interface

The RAG pipeline can answer user questions using the retrieved document context and a local Ollama LLM.

Each response includes the retrieved document sources and page numbers.